In [ ]:
import numpy as np
import tempfile, os
from dnb import Pipeline, FileSource, PipelineConfig, EventType
from dnb.modules import WaveletConvolution, EventDetector, PowerEstimator

# Make a 10-second fake recording with a loud 120 Hz burst on channel 0.
# The burst is at 7-7.2s, well after warmup (15 x 0.2s = 3s).
rng = np.random.default_rng(42)
fs = 30000.0
n = int(10.0 * fs)
t = np.arange(n) / fs
data = rng.standard_normal((4, n)) * 0.5
burst_start = int(7.0 * fs)
burst_end = int(7.2 * fs)
data[0, burst_start:burst_end] += 15 * np.sin(2 * np.pi * 120 * t[burst_start:burst_end])

path = os.path.join(tempfile.mkdtemp(), "test.npz")
np.savez(path, continuous=data, sample_rate=fs)

events = Pipeline(
    source=FileSource(path),
    modules=[
        WaveletConvolution(freq_min=10, freq_max=200, n_freqs=15),
        PowerEstimator(),
        EventDetector(
            event_type=EventType.RIPPLE,
            freq_range=(80, 200),
            threshold_std=4.0,
            min_duration=0.025,
            cooldown=0.15,
            warmup_chunks=15,
        ),
    ],
    config=PipelineConfig(sample_rate=fs, n_channels=4, chunk_duration=0.2),
).run_offline()

print(f"Detected {len(events)} events")
for e in events[:10]:
    print(f"  {e.event_type.name} ch={e.channel_id} t={e.timestamp:.3f}s dur={e.duration:.3f}s")